# Vamos a resolver el ejemplo de viaje por las ciudades de Rumanía

Busca el camino más corto entre Arad y Bucarest. Para ello, crea un grafo con las ciudades y las distancias entre ellas. Encontrarás las distancias en el diccionario `distancies`, que ya está creado.

Utiliza la metodología que hemos visto en clase: crea el estado `EstatRomania` y el problema `ProblemaRomania`. Después, resuelve el problema con UCS y A*.

Te proporcionamos la clase `Problema`, que ya hemos utilizado, el diccionario de distancias y otro diccionario con las distancias en línea recta entre las ciudades y Bucarest.

In [1]:
distancies = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Drobeta': 75},
    'Drobeta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Drobeta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucharest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucharest': 101},
    'Bucharest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90},
    'Giurgiu': {'Bucharest': 90}
}

distancies_recta = {
    'Arad': 366,
    'Zerind': 374,
    'Oradea': 380,
    'Sibiu': 253,
    'Timisoara': 329,
    'Lugoj': 244,
    'Mehadia': 241,
    'Drobeta': 242,
    'Craiova': 160,
    'Rimnicu Vilcea': 193,
    'Fagaras': 176,
    'Pitesti': 100,
    'Bucharest': 0,
    'Giurgiu': 77
}


In [2]:
class Problema(object):
    """Aquesta és la classe abstracta per a un problema formal. Una nova àrea crea una subclasse d'aquesta, sobrescrivint `accions` i `accio`, i potser altres mètodes.
    L'heurística per defecte és 0 i el cost d'acció per defecte és 1 per a tots els estats.
    Quan crees una instància d'una subclasse, especifica els estats `inicial` i `final`
    (o proporciona un mètode `es_resultat`) i potser altres arguments de paraula clau per a la subclasse."""

    def __init__(self, inicial=None, final=None, **kwds):
        self.__dict__.update(inicial=inicial, final=final, **kwds)

    def accions(self, state):         raise NotImplementedError

    def accio(self, state, action):   raise NotImplementedError

    def es_resultat(self, state):     return state == self.final

    def cost_accio(self, s, a, s1):   return 1

    def h(self, estat):               return 0

    def __str__(self):
        return '{}({!r}, {!r})'.format(
            type(self).__name__, self.inicial, self.final)

In [3]:
from dataclasses import dataclass


@dataclass
class EstatRomania(object):
    """Aquesta classe representa un estat del problema de Romania. L'estat és una ciutat."""

    ciutat: str
    pare: 'EstatRomania' = None

    def __repr__(self):
        return 'EstatRomania({!r})'.format(self.ciutat)

    def __str__(self):
        return '{}'.format(self.ciutat)

    def __eq__(self, other):
        return self.ciutat == other.ciutat

    def __hash__(self):
        return hash(self.ciutat)

    def __lt__(self, other):
        return self.ciutat < other.ciutat

In [4]:
class ProblemaRomania(Problema):
    """Aquesta classe representa el problema de Romania. L'estat és una ciutat."""

    def __init__(self, inicial, final):
        super().__init__(inicial, final)

    def accions(self, estat):
        """Retorna una llista d'accions possibles a l'estat actual."""
        return list(distancies[estat.ciutat].keys())

    def accio(self, estat, accio):
        """Retorna l'estat que resulta d'aplicar l'accio a l'estat."""
        return EstatRomania(accio, estat)

    def cost_accio(self, estat, accio, estat1):
        """Retorna el cost d'aplicar l'accio a l'estat."""
        return distancies[estat.ciutat][accio]

    def h(self, estat):
        """Retorna una estimació heurística del cost per anar de l'estat a l'estat final."""
        return distancies_recta[estat.ciutat]

In [5]:
import heapq

def ucs(problema):
    """Algorisme de cerca uniforme. Retorna una llista amb el camí des de l'estat inicial fins a l'estat final."""
    frontera = [(0, problema.inicial)]
    visitats = set()
    while frontera:
        cost, estat = heapq.heappop(frontera)
        if problema.es_resultat(estat):
            print('Visitats:', len(visitats))
            return estat
        if estat not in visitats:
            visitats.add(estat)
            for accio in problema.accions(estat):
                estat1 = problema.accio(estat, accio)
                cost1 = cost + problema.cost_accio(estat, accio, estat1)
                heapq.heappush(frontera, (cost1, estat1))
        

In [6]:
def extreu_ruta(estat):
    """Retorna una llista amb els estats de la ruta des de l'estat inicial fins a l'estat final."""
    ruta = []
    while estat.pare:
        ruta.append(estat)
        estat = estat.pare
    ruta.append(estat)
    ruta.reverse()
    return ruta

def mostra_ruta(estat):
    """Mostra la ruta des de l'estat inicial fins a l'estat final."""
    ruta = extreu_ruta(estat)
    cost = 0
    for estat in ruta:
        if estat.pare:
            print(f"{estat.pare} -> {estat}  ({distancies[estat.pare.ciutat][estat.ciutat]})")
            cost += distancies[estat.pare.ciutat][estat.ciutat]
        else:
            print(estat)

    print(f"Cost: {cost}")

In [7]:
problema = ProblemaRomania(EstatRomania('Arad'), EstatRomania('Bucharest'))
estat = ucs(problema)

mostra_ruta(estat)

Visitats: 12
Arad
Arad -> Sibiu  (140)
Sibiu -> Rimnicu Vilcea  (80)
Rimnicu Vilcea -> Pitesti  (97)
Pitesti -> Bucharest  (101)
Cost: 418


In [ ]:
def a_estrella(problema):
    """Algorisme de cerca A*. Retorna una llista amb el camí des de l'estat inicial fins a l'estat final."""
    frontera = [(problema.h(problema.inicial), 0, problema.inicial)]
    visitats = set()
    while frontera:
        _, cost_actual, estat = heapq.heappop(frontera)

        if estat in visitats:
            continue

        visitats.add(estat)

        if problema.es_resultat(estat):
            print('Visitats:', len(visitats))
            return estat

        for accio in problema.accions(estat):
            nou_estat = problema.accio(estat, accio)
            nou_cost = cost_actual + problema.cost_accio(estat, accio, nou_estat)
            prioritat = nou_cost + problema.h(nou_estat)

            heapq.heappush(frontera, (prioritat, nou_cost, nou_estat))


In [13]:
problema = ProblemaRomania(EstatRomania('Arad'), EstatRomania('Bucharest'))
estat = a_estrella(problema)

mostra_ruta(estat)


Visitats: 6
Arad
Arad -> Sibiu  (140)
Sibiu -> Rimnicu Vilcea  (80)
Rimnicu Vilcea -> Pitesti  (97)
Pitesti -> Bucharest  (101)
Cost: 418


# Ejercicio: A* ponderado

A* utiliza la prioridad $f(n) = g(n) + h(n)$. Vamos a investigar qué ocurre si damos más peso a la heurística:

$$f_w(n) = g(n) + w h(n)$$

- $w = 0$: búsqueda de coste uniforme.
- $w = 1$: A* estándar.
- $w > 1$: A* ponderado; prioriza más la aproximación al destino.

Copia `a_estrella` en la celda siguiente con el nombre `a_estrella_ponderada(problema, pes=1)`. Solo tienes que modificar dos prioridades: la del estado inicial y la de cada sucesor. En ambas, multiplica la heurística por `pes`.

In [ ]:
# TODO: copia ací a_estrella i canvia-la a a_estrella_ponderada(problema, pes=1).
# Recorda: prioritat = g(n) + pes * h(n).


In [ ]:
# Executa aquesta cel·la després de completar a_estrella_ponderada.
for pes in (0, 1, 2, 5, 100):
    problema = ProblemaRomania(EstatRomania('Arad'), EstatRomania('Bucharest'))
    estat = a_estrella_ponderada(problema, pes)
    print(f'Pes {pes}')
    mostra_ruta(estat)
    print()


# Preguntas

Completa una tabla con la ruta, el coste y el número de estados visitados para cada peso:

| Peso $w$ | Ruta | Coste | Estados visitados | ¿Es óptima? |
| ---: | --- | ---: | ---: | --- |
| 0 | | | | |
| 1 | | | | |
| 2 | | | | |
| 5 | | | | |
| 100 | | | | |

1. ¿Qué valor de $w$ corresponde a UCS? ¿Y a A* estándar?
2. ¿A partir de qué peso deja de encontrar el coste óptimo 418?
3. ¿Por qué aumentar $w$ puede reducir los estados explorados?
4. ¿En qué tipo de aplicación aceptarías una solución rápida, aunque no sea óptima? ¿Y en cuál no la aceptarías?